In [ ]:
# crawl_dwts_external_data.py
# -*- coding: utf-8 -*-
"""
Fetch 4 strongly recommended external datasets for DWTS MCM C:
1) season-week air_date (episode list)
2) theme / dance_style / special_rule (if available)
3) Google Trends (weekly attention proxy)
4) Wikipedia Pageviews (daily -> aggregate to season-week)

Outputs:
- episode_meta.csv
- trends_weekly.csv
- pageviews_weekly.csv
- weekly_panel_enriched.csv
"""

import os
import re
import time
import json
import math
import random
import logging
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
from dateutil import parser

# Optional, only used if you enable Google Trends fetching:
# pip install pytrends
try:
    from pytrends.request import TrendReq
except Exception:
    TrendReq = None


# -----------------------
# Config
# -----------------------
@dataclass
class Config:
    official_csv: str = "2026_MCM_Problem_C_Data.csv"
    out_dir: str = "external_outputs"
    user_agent: str = "DWTS-MCM-Crawler/1.0 (contact: your_email@example.com)"
    sleep_base: float = 0.9              # base sleep for polite crawling
    sleep_jitter: float = 0.5            # random jitter
    max_retries: int = 3
    timeout_sec: int = 30

    # Google Trends
    enable_trends: bool = True
    trends_geo: str = "US"
    trends_keyword_prefix: str = "DWTS "  # reduce name ambiguity
    trends_pause_sec: float = 1.2         # extra pause between trend calls

    # Pageviews
    enable_pageviews: bool = True
    pageviews_window_days: int = 3        # aggregate to week by air_date ± window days
    pageviews_project: str = "en.wikipedia.org"
    pageviews_access: str = "all-access"
    pageviews_agent: str = "user"

    # Cache
    cache_dir: str = "cache"
    reuse_cache: bool = True


# -----------------------
# Logging
# -----------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

# -----------------------
# HTTP helpers
# -----------------------
def make_session(cfg: Config) -> requests.Session:
    s = requests.Session()
    s.headers.update({"User-Agent": cfg.user_agent})
    return s

def polite_sleep(cfg: Config, extra: float = 0.0) -> None:
    time.sleep(cfg.sleep_base + random.random() * cfg.sleep_jitter + extra)

def request_json(session: requests.Session, cfg: Config, url: str, params: dict) -> dict:
    last_err = None
    for attempt in range(cfg.max_retries):
        try:
            r = session.get(url, params=params, timeout=cfg.timeout_sec)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            polite_sleep(cfg, extra=1.0 + attempt)
    raise RuntimeError(f"Failed request_json after {cfg.max_retries} retries: {url}, err={last_err}")

def request_text(session: requests.Session, cfg: Config, url: str) -> str:
    last_err = None
    for attempt in range(cfg.max_retries):
        try:
            r = session.get(url, timeout=cfg.timeout_sec)
            r.raise_for_status()
            return r.text
        except Exception as e:
            last_err = e
            polite_sleep(cfg, extra=1.0 + attempt)
    raise RuntimeError(f"Failed request_text after {cfg.max_retries} retries: {url}, err={last_err}")


# -----------------------
# Step 0: read official data & build weekly panel base
# -----------------------
def build_weekly_panel_base(official_csv: str) -> pd.DataFrame:
    df = pd.read_csv(official_csv)

    # Find weekX_judgeY columns
    pat = re.compile(r"week(\d+)_judge(\d+)_", re.I)
    score_cols = [c for c in df.columns if pat.search(c)]

    long = df.melt(
        id_vars=[
            "season", "celebrity_name", "ballroom_partner",
            "celebrity_industry", "celebrity_homestate",
            "celebrity_homecountry/region", "celebrity_age_during_season",
            "results", "placement"
        ],
        value_vars=score_cols,
        var_name="wk_jg",
        value_name="score"
    )
    long["week"] = long["wk_jg"].str.extract(r"week(\d+)", flags=re.I).astype(int)
    long["judge"] = long["wk_jg"].str.extract(r"judge(\d+)", flags=re.I).astype(int)
    long = long.drop(columns=["wk_jg"])

    panel = (long.groupby(["season","week","celebrity_name"], as_index=False)["score"]
             .sum(min_count=1)
             .rename(columns={"score":"judge_total"}))

    panel["judge_rank"] = panel.groupby(["season","week"])["judge_total"].rank(ascending=False, method="min")
    panel["judge_share"] = panel["judge_total"] / panel.groupby(["season","week"])["judge_total"].transform("sum")

    return panel


# -----------------------
# Step 1: find Wikipedia season page via MediaWiki API (reliable, avoids hardcoding URLs)
# -----------------------
def wiki_search_season_page(session: requests.Session, cfg: Config, season: int) -> Optional[str]:
    """
    Use MediaWiki API search to find the most likely page title for:
    "Dancing with the Stars (American season X)"
    Returns full URL like https://en.wikipedia.org/wiki/<Title>
    """
    api = "https://en.wikipedia.org/w/api.php"
    query = f"Dancing with the Stars (American season {season})"
    data = request_json(session, cfg, api, params={
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": 5
    })
    items = data.get("query", {}).get("search", [])
    if not items:
        return None

    # pick the top result
    title = items[0].get("title")
    if not title:
        return None

    # convert to canonical URL
    # (Spaces -> underscores)
    title_url = title.replace(" ", "_")
    return f"https://en.wikipedia.org/wiki/{requests.utils.quote(title_url)}"


# -----------------------
# Step 2: parse episode/week table from season page
# -----------------------
def parse_date_safe(x) -> Optional[pd.Timestamp]:
    if pd.isna(x):
        return None
    try:
        return pd.to_datetime(parser.parse(str(x), fuzzy=True))
    except Exception:
        return None

def select_best_table(tables: List[pd.DataFrame]) -> Optional[pd.DataFrame]:
    """
    Heuristic: pick a table likely representing week/episode info.
    We prefer tables containing columns like Week and Air date/Date.
    """
    best = None
    best_score = -1
    for t in tables:
        cols = [str(c).lower() for c in t.columns]
        score = 0
        if any("week" in c for c in cols):
            score += 3
        if any("air date" in c or c == "date" or "date" in c for c in cols):
            score += 3
        if any("theme" in c for c in cols):
            score += 1
        if any("dance" in c for c in cols):
            score += 1
        if score > best_score:
            best_score = score
            best = t
    return best

def fetch_episode_meta_for_season(session: requests.Session, cfg: Config, season: int) -> pd.DataFrame:
    url = wiki_search_season_page(session, cfg, season)
    if not url:
        raise RuntimeError(f"Season {season}: cannot find Wikipedia page via search.")

    logging.info(f"[EpisodeMeta] Season {season} page: {url}")
    html = request_text(session, cfg, url)
    tables = pd.read_html(html)

    best = select_best_table(tables)
    if best is None:
        raise RuntimeError(f"Season {season}: no tables found.")

    df = best.copy()
    df.columns = [str(c).strip() for c in df.columns]

    # rename likely columns
    col_map = {}
    for c in df.columns:
        cl = c.lower()
        if "week" in cl:
            col_map[c] = "week_raw"
        elif "air date" in cl or cl == "date" or "date" in cl:
            col_map[c] = "air_date_raw"
        elif "theme" in cl:
            col_map[c] = "theme"
        elif "dance" in cl:
            col_map[c] = "dance_style"
        elif "special" in cl or "twist" in cl:
            col_map[c] = "special_rule"
    df = df.rename(columns=col_map)
    df["season"] = season

    # parse week
    if "week_raw" in df.columns:
        df["week"] = pd.to_numeric(df["week_raw"], errors="coerce")
    else:
        df["week"] = np.nan

    # parse air_date
    if "air_date_raw" in df.columns:
        df["air_date"] = df["air_date_raw"].apply(parse_date_safe)
    else:
        df["air_date"] = pd.NaT

    # if week missing, derive from air_date order
    if df["week"].isna().all():
        df = df.sort_values("air_date")
        df["week"] = range(1, len(df) + 1)

    # basic cleanup: keep reasonable week values
    df = df.dropna(subset=["week"])
    df["week"] = df["week"].astype(int)

    keep = ["season", "week"]
    for c in ["air_date", "theme", "dance_style", "special_rule"]:
        if c in df.columns:
            keep.append(c)
    out = df[keep].drop_duplicates(subset=["season", "week"]).sort_values(["season", "week"]).reset_index(drop=True)

    polite_sleep(cfg)
    return out


# -----------------------
# Step 3: map any date to season-week using air_date ± window
# -----------------------
def build_week_windows(ep: pd.DataFrame, window_days: int) -> pd.DataFrame:
    """
    Build date windows [air_date - window_days, air_date + window_days] for each season-week.
    """
    e = ep.copy()
    if "air_date" not in e.columns:
        raise ValueError("episode_meta must include air_date for window mapping.")
    e = e.dropna(subset=["air_date"]).copy()
    e["win_start"] = e["air_date"] - pd.to_timedelta(window_days, unit="D")
    e["win_end"] = e["air_date"] + pd.to_timedelta(window_days, unit="D")
    return e[["season","week","air_date","win_start","win_end"]]

def map_date_to_week(date: pd.Timestamp, windows: pd.DataFrame) -> Optional[int]:
    """
    Given a date and windows for one season, return matching week if date falls into any window.
    If none, return nearest week by absolute difference to air_date.
    """
    w = windows
    m = w[(date >= w["win_start"]) & (date <= w["win_end"])]
    if len(m) > 0:
        # if multiple windows overlap (rare), take closest air_date
        idx = (m["air_date"] - date).abs().idxmin()
        return int(m.loc[idx, "week"])
    # fallback: nearest air_date
    idx = (w["air_date"] - date).abs().idxmin()
    return int(w.loc[idx, "week"])


# -----------------------
# Step 4: Wikipedia Pageviews (daily -> season-week aggregate)
# -----------------------
def wiki_search_page_title(session: requests.Session, cfg: Config, query: str) -> Optional[str]:
    """
    Find a likely Wikipedia page title for a celebrity.
    This is heuristic; for best results, maintain a manual mapping for ambiguous names.
    """
    api = "https://en.wikipedia.org/w/api.php"
    data = request_json(session, cfg, api, params={
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": 5
    })
    items = data.get("query", {}).get("search", [])
    if not items:
        return None
    title = items[0].get("title")
    return title

def fetch_pageviews_daily(session: requests.Session, cfg: Config, page_title: str,
                         start_yyyymmdd: str, end_yyyymmdd: str) -> pd.DataFrame:
    """
    Wikimedia Pageviews API:
    https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/{project}/{access}/{agent}/{article}/{granularity}/{start}/{end}
    """
    base = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article"
    article = requests.utils.quote(page_title.replace(" ", "_"))
    url = f"{base}/{cfg.pageviews_project}/{cfg.pageviews_access}/{cfg.pageviews_agent}/{article}/daily/{start_yyyymmdd}/{end_yyyymmdd}"

    last_err = None
    for attempt in range(cfg.max_retries):
        try:
            r = session.get(url, timeout=cfg.timeout_sec)
            if r.status_code == 404:
                return pd.DataFrame()
            r.raise_for_status()
            items = r.json().get("items", [])
            if not items:
                return pd.DataFrame()
            df = pd.DataFrame(items)
            df["date"] = pd.to_datetime(df["timestamp"].str.slice(0, 8), format="%Y%m%d")
            df = df.rename(columns={"views":"pageviews"})
            return df[["date","pageviews"]]
        except Exception as e:
            last_err = e
            polite_sleep(cfg, extra=1.0 + attempt)

    raise RuntimeError(f"Pageviews failed for {page_title}, err={last_err}")

def yyyymmdd(dt: pd.Timestamp) -> str:
    return dt.strftime("%Y%m%d")

def aggregate_pageviews_to_season_week(pv_daily: pd.DataFrame, season: int,
                                      windows_one_season: pd.DataFrame) -> pd.DataFrame:
    if pv_daily.empty:
        return pd.DataFrame()

    rows = []
    for _, r in pv_daily.iterrows():
        d = r["date"]
        w = map_date_to_week(d, windows_one_season)
        rows.append((season, w, int(r["pageviews"])))
    out = pd.DataFrame(rows, columns=["season","week","pageviews"])
    out = out.groupby(["season","week"], as_index=False)["pageviews"].sum()
    return out


# -----------------------
# Step 5: Google Trends (weekly attention proxy)
# -----------------------
def fetch_trends(session_cfg: Config, keyword: str, start_date: pd.Timestamp, end_date: pd.Timestamp) -> pd.DataFrame:
    if TrendReq is None:
        raise RuntimeError("pytrends is not installed. Run: pip install pytrends")

    # Google Trends is sensitive to rate-limits. Use retries & pauses.
    last_err = None
    for attempt in range(session_cfg.max_retries):
        try:
            pytrends = TrendReq(hl="en-US", tz=360, retries=2, backoff_factor=0.3)
            timeframe = f"{start_date.date()} {end_date.date()}"
            pytrends.build_payload([keyword], timeframe=timeframe, geo=session_cfg.trends_geo)
            df = pytrends.interest_over_time()
            if df.empty:
                return df
            df = df.reset_index().rename(columns={keyword: "trend_index"})
            df = df[["date","trend_index"]]
            return df
        except Exception as e:
            last_err = e
            time.sleep(session_cfg.trends_pause_sec + 1.5 * (attempt + 1))

    raise RuntimeError(f"Trends failed for keyword={keyword}, err={last_err}")

def aggregate_trends_to_season_week(tr_daily_or_weekly: pd.DataFrame, season: int,
                                   windows_one_season: pd.DataFrame) -> pd.DataFrame:
    """
    pytrends may return daily or weekly. We map each returned date to season-week and average.
    """
    if tr_daily_or_weekly.empty:
        return pd.DataFrame()

    rows = []
    for _, r in tr_daily_or_weekly.iterrows():
        d = pd.to_datetime(r["date"])
        w = map_date_to_week(d, windows_one_season)
        rows.append((season, w, float(r["trend_index"])))
    out = pd.DataFrame(rows, columns=["season","week","trend_index"])
    out = out.groupby(["season","week"], as_index=False)["trend_index"].mean()
    return out


# -----------------------
# Main pipeline
# -----------------------
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def cache_path(cfg: Config, name: str) -> str:
    ensure_dir(cfg.cache_dir)
    return os.path.join(cfg.cache_dir, name)

def load_or_run(cfg: Config, path: str, fn):
    if cfg.reuse_cache and os.path.exists(path):
        logging.info(f"Using cache: {path}")
        return pd.read_csv(path)
    df = fn()
    df.to_csv(path, index=False)
    logging.info(f"Cached: {path}")
    return df

def main():
    cfg = Config()
    ensure_dir(cfg.out_dir)
    session = make_session(cfg)

    # Build weekly panel base
    weekly_panel = build_weekly_panel_base(cfg.official_csv)

    seasons = sorted(weekly_panel["season"].unique().tolist())
    logging.info(f"Found seasons in official data: {seasons[0]}..{seasons[-1]} (n={len(seasons)})")

    # ---- Fetch episode meta (air_date/theme/dance/special_rule when available)
    def run_episode_meta():
        all_eps = []
        for s in seasons:
            try:
                ep = fetch_episode_meta_for_season(session, cfg, s)
                all_eps.append(ep)
            except Exception as e:
                logging.warning(f"Episode meta failed for season {s}: {e}")
            polite_sleep(cfg)
        if not all_eps:
            return pd.DataFrame(columns=["season","week","air_date","theme","dance_style","special_rule"])
        out = pd.concat(all_eps, ignore_index=True)
        # normalize
        if "air_date" in out.columns:
            out["air_date"] = pd.to_datetime(out["air_date"], errors="coerce")
        return out

    episode_meta_cache = cache_path(cfg, "episode_meta.csv")
    episode_meta = load_or_run(cfg, episode_meta_cache, run_episode_meta)

    # Save episode_meta
    episode_meta_out = os.path.join(cfg.out_dir, "episode_meta.csv")
    episode_meta.to_csv(episode_meta_out, index=False)
    logging.info(f"Saved: {episode_meta_out}")

    # Merge episode_meta into weekly_panel
    merged = weekly_panel.merge(episode_meta, on=["season","week"], how="left")

    # Build week windows for mapping dates -> week
    if "air_date" not in episode_meta.columns or episode_meta["air_date"].isna().all():
        logging.warning("No air_date parsed. Trends/Pageviews mapping may be unreliable.")
        windows = None
    else:
        windows = build_week_windows(episode_meta, cfg.pageviews_window_days)

    # ---- Pageviews (optional)
    pv_rows = []
    if cfg.enable_pageviews and windows is not None:
        # seasons with valid windows
        for s in seasons:
            win_s = windows[windows["season"] == s].copy()
            if win_s.empty:
                continue

            # determine season time range for API call
            start_dt = win_s["win_start"].min() - pd.to_timedelta(2, unit="D")
            end_dt = win_s["win_end"].max() + pd.to_timedelta(2, unit="D")
            start_str, end_str = yyyymmdd(start_dt), yyyymmdd(end_dt)

            # pick celebrities of that season
            celebs = (weekly_panel[weekly_panel["season"] == s]["celebrity_name"]
                      .drop_duplicates().tolist())

            for name in celebs:
                # caching per season+name
                cache_file = cache_path(cfg, f"pageviews_{s}_{re.sub(r'[^A-Za-z0-9]+','_',name)[:40]}.csv")

                def run_one():
                    # Search wiki title (heuristic). For better results, manually map ambiguous names.
                    title = wiki_search_page_title(session, cfg, name)
                    if not title:
                        return pd.DataFrame()
                    daily = fetch_pageviews_daily(session, cfg, title, start_str, end_str)
                    if daily.empty:
                        return pd.DataFrame()
                    wk = aggregate_pageviews_to_season_week(daily, s, win_s)
                    wk["celebrity_name"] = name
                    wk["wiki_title"] = title
                    return wk

                try:
                    wk = load_or_run(cfg, cache_file, run_one)
                    if not wk.empty:
                        pv_rows.append(wk)
                except Exception as e:
                    logging.warning(f"Pageviews failed: season={s}, name={name}, err={e}")

                polite_sleep(cfg, extra=0.2)

    pageviews_weekly = pd.concat(pv_rows, ignore_index=True) if pv_rows else pd.DataFrame(
        columns=["season","week","pageviews","celebrity_name","wiki_title"]
    )
    pv_out = os.path.join(cfg.out_dir, "pageviews_weekly.csv")
    pageviews_weekly.to_csv(pv_out, index=False)
    logging.info(f"Saved: {pv_out}")

    # ---- Google Trends (optional)
    tr_rows = []
    if cfg.enable_trends and windows is not None:
        for s in seasons:
            win_s = windows[windows["season"] == s].copy()
            if win_s.empty:
                continue

            # season window
            start_dt = win_s["win_start"].min()
            end_dt = win_s["win_end"].max()

            celebs = (weekly_panel[weekly_panel["season"] == s]["celebrity_name"]
                      .drop_duplicates().tolist())

            for name in celebs:
                cache_file = cache_path(cfg, f"trends_{s}_{re.sub(r'[^A-Za-z0-9]+','_',name)[:40]}.csv")

                def run_one_tr():
                    kw = f"{cfg.trends_keyword_prefix}{name}".strip()
                    td = fetch_trends(cfg, kw, start_dt, end_dt)
                    if td.empty:
                        return pd.DataFrame()
                    wk = aggregate_trends_to_season_week(td, s, win_s)
                    wk["celebrity_name"] = name
                    wk["keyword"] = kw
                    return wk

                try:
                    wk = load_or_run(cfg, cache_file, run_one_tr)
                    if not wk.empty:
                        tr_rows.append(wk)
                except Exception as e:
                    logging.warning(f"Trends failed: season={s}, name={name}, err={e}")

                time.sleep(cfg.trends_pause_sec + random.random()*0.3)

    trends_weekly = pd.concat(tr_rows, ignore_index=True) if tr_rows else pd.DataFrame(
        columns=["season","week","trend_index","celebrity_name","keyword"]
    )
    tr_out = os.path.join(cfg.out_dir, "trends_weekly.csv")
    trends_weekly.to_csv(tr_out, index=False)
    logging.info(f"Saved: {tr_out}")

    # ---- Merge external data into weekly panel
    if not pageviews_weekly.empty:
        merged = merged.merge(
            pageviews_weekly[["season","week","celebrity_name","pageviews"]],
            on=["season","week","celebrity_name"], how="left"
        )
    if not trends_weekly.empty:
        merged = merged.merge(
            trends_weekly[["season","week","celebrity_name","trend_index"]],
            on=["season","week","celebrity_name"], how="left"
        )

    out_final = os.path.join(cfg.out_dir, "weekly_panel_enriched.csv")
    merged.to_csv(out_final, index=False)
    logging.info(f"Saved: {out_final}")
    logging.info("Done.")


if __name__ == "__main__":
    main()
